# Time travel on silver `empresas` (RF-06)

Silver is the current state of every business key, loaded by a Delta `MERGE`
([ADR 011](../docs/adr/011-silver-current-state-merge.md)). Every load commits **one Delta
version**, so the state right after an earlier load is still readable: `versionAsOf` (by
version number) or `timestampAsOf` (by commit time).

**Acceptance (RF-06):** the result of a query on version N-1 can be reproduced.

**Where the outputs come from.** The lake lives in MinIO inside the CI `bronze` job (ADR 006;
no real data on the development laptop), so this notebook is not executed locally. The code
cells are what `cnpj_lakehouse.time_travel` runs; the outputs were pasted from the CI report
`build/time-travel/time-travel.json` of [this run](https://github.com/thalesreiser/spark-cnpj-lakehouse/actions/runs/39330744721) (lookups + `Empresas1` of
2026-09, then a synthetic 2026-10 derived from it), formatted for reading. Run it yourself with `make up`, `make
silver` twice (two months) and `make time-travel`. Only aggregates and metadata are shown,
never rows, names or CNPJs (RST-01/LGPD).

In [1]:
import json
import os

from cnpj_lakehouse import time_travel as tt
from cnpj_lakehouse.spark import LakehouseConfig, get_spark

cfg = LakehouseConfig.from_env(os.environ)
spark = get_spark("notebook-time-travel", cfg)
path = cfg.path("silver", "empresas")  # s3a://lakehouse/silver/empresas
print(path)

s3a://lakehouse/silver/empresas


## 1. `DESCRIBE HISTORY`

One row per commit: version 0 is the first load (`WRITE`), each later month a `MERGE` with its
metrics. Here: version 0 = 2026-09, version 1 = the re-run of 2026-09 (a `MERGE` that
inserted and updated nothing), version 2 = the synthetic 2026-10.

In [2]:
columns = ("version", "timestamp", "operation", "source_rows", "inserted", "updated")
for c in tt.history(spark, path):  # DESCRIBE HISTORY delta.`<path>`
    print(*(c[k] for k in columns))

version                timestamp  operation  source_rows  inserted  updated
      0  2026-10-19 21:59:45.000      WRITE      4494860   4494860        0
      1  2026-10-19 22:03:49.000      MERGE      4494860         0        0
      2  2026-10-19 22:08:20.000      MERGE      4494601     44680    45012


## 2. `versionAsOf`: N-1 versus N

The same aggregate query on version N-1 = 1 (after 2026-09) and on N =
2 (after 2026-10). Equivalent SQL:
`SELECT porte_empresa, count(*) FROM delta.`<path>` VERSION AS OF 1 GROUP BY 1`.

In [3]:
old = tt.read_version(spark, path, 1)  # .option("versionAsOf", 1)
new = tt.read_version(spark, path, 2)
for s in (tt.snapshot(old), tt.snapshot(new)):
    print(s)
print(tt.changed_keys(old, new, ["cnpj_basico"]))

version 1: rows 4494860, checksum 9249600739857961180549
porte_empresa: {"01": 2164059, "03": 151105, "05": 2179696}
natureza_juridica: 72 codes, top 3 {"2062": 1944471, "2135": 1860132, "3999": 311540}
version 2: rows 4539540, checksum 7356905653008764571409
porte_empresa: {"01": 2185376, "03": 152609, "05": 2201555}
natureza_juridica: 72 codes, top 3 {"2062": 1963846, "2135": 1878585, "3999": 314675}

      v1 (N-1)   v2 (N)  N - (N-1)
rows   4494860  4539540      44680
  01   2164059  2185376      21317
  03    151105   152609       1504
  05   2179696  2201555      21859

changed keys: {"changed": 45012, "only_new": 44680, "only_old": 0}


## 3. `timestampAsOf`

Reading the table "as of" the commit time of version N-1 resolves to that version (Delta picks
the latest commit at or before the timestamp), so it gives the same aggregates.

In [4]:
ts = "2026-10-19 22:03:49.000"  # timestamp of version 1 in DESCRIBE HISTORY
print(tt.snapshot(tt.read_timestamp(spark, path, ts)) == tt.snapshot(old))

True


## 4. Reproducibility check

Right after the 2026-09 load the CI job recorded the aggregates of the current version
(`make time-travel TT_ARGS="--snapshot ..."`). After the 2026-10 load, `versionAsOf N-1` must
give exactly the same rows, checksum and counts (`--expected ...`); the job fails otherwise.

In [5]:
with open("build/time-travel-2026-09.json", encoding="utf-8") as f:
    recorded = json.load(f)
report = tt.demo(spark, path, "empresas", recorded)
print(report["checks"], report["ok"])

timestamp_as_of_equals_version_as_of: True
version_as_of_reproduces_recorded: True
ok: True


## Interpretation

* The history has the 2026-09 load, its no-op re-run (version 1, 0 inserted,
  0 updated) and one `MERGE` for 2026-10 (version 2) with that month's counts.
* Version N has 44680 more rows than N-1: exactly the
  44680 keys that exist only in N (inserted). 45012 keys have a different
  `_row_hash` (updated); 0 keys disappeared, because the MERGE keeps absent keys.
* Reading N-1 by version or by timestamp gives the same aggregates as the snapshot recorded
  right after that load, so the result of a query on N-1 is reproducible (RF-06).
* Time travel depends on the Delta log and data files of old versions: `VACUUM` with a short
  retention would remove them (not run in this project).